# Cross-Cultural Tourist Review Analysis — Data Overview

**Corpus**: 496 reviews · 8 places (Seoul + Busan) · Korean (domestic) vs English (foreign)

This notebook presents all analysis results from the pipeline (03→04→05→06→07).

In [3]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Korean font support
for fp in ('C:/Windows/Fonts/malgunbd.ttf', 'C:/Windows/Fonts/malgun.ttf'):
    if Path(fp).exists():
        fm.fontManager.addfont(fp)
        plt.rcParams['font.family'] = fm.FontProperties(fname=fp).get_name()
        break

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

DATA = Path('/data/analysis')
CORPUS = Path('/data/corpus.csv')

PLACE_EN = {
    '경복궁': 'Gyeongbokgung', '창덕궁': 'Changdeokgung',
    '북촌한옥마을': 'Bukchon Village', '광장시장': 'Gwangjang Mkt',
    '홍대거리': 'Hongdae St.', '롯데월드': 'Lotte World',
    '광안리해수욕장': 'Gwangalli Beach', '해운대해수욕장': 'Haeundae Beach',
}

print('Ready.')

Ready.


---
## 1. Corpus Summary

In [4]:
df = pd.read_csv(CORPUS, encoding='utf-8-sig')
print(f'Total reviews: {len(df)}')
print(f'Places: {df["place"].nunique()}')
print(f'Languages: {df["language"].unique().tolist()}')
print()

corpus_table = df.groupby(['place','language']).size().unstack(fill_value=0)
corpus_table.index = [f"{PLACE_EN.get(p, p)} ({p})" for p in corpus_table.index]
corpus_table['Total'] = corpus_table.sum(axis=1)
corpus_table.loc['TOTAL'] = corpus_table.sum()
corpus_table

FileNotFoundError: [Errno 2] No such file or directory: '\\data\\corpus.csv'

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
places = list(PLACE_EN.keys())
places = [p for p in places if p in df['place'].unique()]
ko_counts = [len(df[(df.place==p) & (df.language=='ko')]) for p in places]
en_counts = [len(df[(df.place==p) & (df.language=='en')]) for p in places]
x = np.arange(len(places))
w = 0.35
ax.bar(x - w/2, ko_counts, w, label='Korean (KO)', color='#2563EB', alpha=0.8)
ax.bar(x + w/2, en_counts, w, label='English (EN)', color='#DC2626', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([PLACE_EN[p] for p in places], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Number of Reviews')
ax.set_title('Corpus Distribution by Place and Language')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Sentiment Analysis

In [ ]:
sent = pd.read_csv(DATA / 'summary_sentiment.csv')
mw = pd.read_csv(DATA / 'stats_mannwhitney.csv')

# Build sentiment comparison table
sent_wide = sent.pivot(index='place', columns='language', values='mean').rename(
    columns={'ko': 'KO_mean', 'en': 'EN_mean'})
sent_wide = sent_wide.merge(mw[['place','p_value']], on='place', how='left')
sent_wide['Gap (KO-EN)'] = sent_wide['KO_mean'] - sent_wide['EN_mean']
sent_wide['Sig'] = sent_wide['p_value'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.')))
sent_wide.index = [f"{PLACE_EN.get(p, p)}" for p in sent_wide['place']]
sent_wide[['KO_mean','EN_mean','Gap (KO-EN)','p_value','Sig']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
places_in = [p for p in PLACE_EN if p in sent_wide['place'].values]
ko_vals = [float(sent_wide[sent_wide.place==p]['KO_mean']) for p in places_in]
en_vals = [float(sent_wide[sent_wide.place==p]['EN_mean']) for p in places_in]
x = np.arange(len(places_in))
w = 0.35
ax.bar(x - w/2, ko_vals, w, label='Korean (KO)', color='#2563EB', alpha=0.8)
ax.bar(x + w/2, en_vals, w, label='English (EN)', color='#DC2626', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([PLACE_EN[p] for p in places_in], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Mean Sentiment Score (0–1)')
ax.set_title('Sentiment Comparison: Korean vs English Reviewers')
ax.set_ylim(0, 1)
ax.legend()

# mark significant places
for i, p in enumerate(places_in):
    pval = float(sent_wide[sent_wide.place==p]['p_value'])
    if pval < 0.05:
        ax.annotate('*', (i, max(ko_vals[i], en_vals[i]) + 0.02),
                    ha='center', fontsize=16, fontweight='bold', color='#16a34a')
plt.tight_layout()
plt.show()

---
## 3. Cross-lingual Semantic Similarity

In [ ]:
sim = pd.read_csv(DATA / 'crosslingual_similarity.csv')
sim.index = [PLACE_EN.get(p, p) for p in sim['place']]
sim[['ko_en_sim','ko_intra_sim','en_intra_sim']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
places_s = [p for p in PLACE_EN if p in sim['place'].values]
x = np.arange(len(places_s))
vals = [float(sim[sim.place==p]['ko_en_sim']) for p in places_s]
ko_intra = [float(sim[sim.place==p]['ko_intra_sim']) for p in places_s]
en_intra = [float(sim[sim.place==p]['en_intra_sim']) for p in places_s]

ax.bar(x - 0.25, vals, 0.25, label='KO↔EN (cross)', color='#16a34a', alpha=0.8)
ax.bar(x, ko_intra, 0.25, label='KO intra', color='#2563EB', alpha=0.6)
ax.bar(x + 0.25, en_intra, 0.25, label='EN intra', color='#DC2626', alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels([PLACE_EN[p] for p in places_s], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Cosine Similarity')
ax.set_title('Cross-lingual vs Intra-group Semantic Similarity')
ax.set_ylim(0.75, 0.95)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Travel Motivation (Zero-shot Classification)

In [ ]:
mot = pd.read_csv(DATA / 'motivation_summary.csv')

# Overall motivation by language
overall = mot.groupby(['language','motivation'])['count'].sum().reset_index()
totals = overall.groupby('language')['count'].transform('sum')
overall['pct'] = (overall['count'] / totals * 100).round(1)
mot_wide = overall.pivot(index='motivation', columns='language', values='pct').fillna(0)
mot_wide.columns = ['EN %', 'KO %']
mot_wide['Gap'] = mot_wide['KO %'] - mot_wide['EN %']
mot_wide.sort_values('KO %', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
motivations = mot_wide.sort_values('KO %', ascending=True).index.tolist()
y = np.arange(len(motivations))
h = 0.35
ax.barh(y - h/2, [mot_wide.loc[m, 'KO %'] for m in motivations], h,
        label='Korean (KO)', color='#2563EB', alpha=0.8)
ax.barh(y + h/2, [mot_wide.loc[m, 'EN %'] for m in motivations], h,
        label='English (EN)', color='#DC2626', alpha=0.8)
ax.set_yticks(y)
ax.set_yticklabels(motivations, fontsize=9)
ax.set_xlabel('Percentage (%)')
ax.set_title('Travel Motivation Distribution')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Lexical Diversity (TTR)

In [ ]:
ttr = pd.read_csv(DATA / 'ttr_results.csv')
ttr_wide = ttr.pivot(index='place', columns='language', values='mean')
ttr_wide.columns = ['EN_TTR', 'KO_TTR']
ttr_wide['Gap'] = ttr_wide['KO_TTR'] - ttr_wide['EN_TTR']
ttr_wide.index = [PLACE_EN.get(p, p) for p in ttr_wide.index]
ttr_wide

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
places_t = [p for p in PLACE_EN if p in ttr.place.unique()]
x = np.arange(len(places_t))
ko_ttr = [float(ttr[(ttr.place==p)&(ttr.language=='ko')]['mean']) for p in places_t]
en_ttr = [float(ttr[(ttr.place==p)&(ttr.language=='en')]['mean']) for p in places_t]
w = 0.35
ax.bar(x - w/2, ko_ttr, w, label='Korean (KO)', color='#2563EB', alpha=0.8)
ax.bar(x + w/2, en_ttr, w, label='English (EN)', color='#DC2626', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([PLACE_EN[p] for p in places_t], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Type-Token Ratio')
ax.set_title('Lexical Diversity (TTR) by Place and Language')
ax.set_ylim(0.75, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. TF-IDF Keyword Contrast

In [ ]:
tfidf = pd.read_csv(DATA / 'tfidf_contrast.csv')

ko_top = tfidf[tfidf.language=='ko'].sort_values('mean_tfidf', ascending=False).head(15)
en_top = tfidf[tfidf.language=='en'].sort_values('mean_tfidf', ascending=False).head(15)

print('=== Top 15 Korean Keywords ===')
for i, (_, r) in enumerate(ko_top.iterrows(), 1):
    print(f"  {i:2d}. {r['word']:12s}  (TF-IDF: {r['mean_tfidf']:.4f})")

print()
print('=== Top 15 English Keywords ===')
for i, (_, r) in enumerate(en_top.iterrows(), 1):
    print(f"  {i:2d}. {r['word']:12s}  (TF-IDF: {r['mean_tfidf']:.4f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ko10 = ko_top.head(10)
ax1.barh(range(len(ko10)), ko10['mean_tfidf'].values, color='#2563EB', alpha=0.8)
ax1.set_yticks(range(len(ko10)))
ax1.set_yticklabels(ko10['word'].values)
ax1.invert_yaxis()
ax1.set_xlabel('Mean TF-IDF')
ax1.set_title('Korean — Top Keywords')

en10 = en_top.head(10)
ax2.barh(range(len(en10)), en10['mean_tfidf'].values, color='#DC2626', alpha=0.8)
ax2.set_yticks(range(len(en10)))
ax2.set_yticklabels(en10['word'].values)
ax2.invert_yaxis()
ax2.set_xlabel('Mean TF-IDF')
ax2.set_title('English — Top Keywords')

plt.tight_layout()
plt.show()

---
## 7. LDA Topic Modeling

In [ ]:
lda_en = pd.read_csv(DATA / 'lda_en_topics.csv')
lda_ko = pd.read_csv(DATA / 'lda_ko_topics.csv')

print('=== English LDA Topics (K=5) ===')
for _, r in lda_en.iterrows():
    print(f"  Topic {r['topic']}: {r['top_words']}")

print()
print('=== Korean LDA Topics (K=5) ===')
for _, r in lda_ko.iterrows():
    print(f"  Topic {r['topic']}: {r['top_words']}")

In [ ]:
topics = pd.read_csv(DATA / 'topic_assignments.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, lang, title, color in [(ax1, 'ko', 'Korean Topics', '#2563EB'),
                                (ax2, 'en', 'English Topics', '#DC2626')]:
    sub = topics[topics.language == lang]
    topic_counts = sub['dominant_topic'].value_counts().sort_index()
    ax.bar(topic_counts.index, topic_counts.values, color=color, alpha=0.8)
    ax.set_xlabel('Topic')
    ax.set_ylabel('Number of Reviews')
    ax.set_title(f'{title} — Distribution')
    ax.set_xticks(topic_counts.index)

plt.tight_layout()
plt.show()

---
## 8. ABSA — Aspect-Based Sentiment Analysis

Five tourism dimensions scored by Qwen2-0.5B-Instruct:
- Historical/Cultural Heritage
- Accessibility
- Crowding
- Food
- Visual/Photo Experience

In [ ]:
absa = pd.read_csv(DATA / 'absa_summary.csv')
absa.index = [f"{PLACE_EN.get(r['place'], r['place'])} ({r['language'].upper()})"
              for _, r in absa.iterrows()]

aspect_cols = ['historical_cultural_mean','accessibility_mean','crowding_mean',
               'food_mean','visual_photo_mean']
aspect_labels = ['Historical', 'Accessibility', 'Crowding', 'Food', 'Visual']

absa[aspect_cols].rename(columns=dict(zip(aspect_cols, aspect_labels)))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

places_a = [p for p in PLACE_EN if p in absa['place'].values]

for i, (col, label) in enumerate(zip(aspect_cols, aspect_labels)):
    ax = axes[i]
    ko_vals = [float(absa[(absa.place==p)&(absa.language=='ko')][col].iloc[0])
               if not absa[(absa.place==p)&(absa.language=='ko')].empty else 0
               for p in places_a]
    en_vals = [float(absa[(absa.place==p)&(absa.language=='en')][col].iloc[0])
               if not absa[(absa.place==p)&(absa.language=='en')].empty else 0
               for p in places_a]
    x = np.arange(len(places_a))
    w = 0.35
    ax.bar(x - w/2, ko_vals, w, label='KO', color='#2563EB', alpha=0.8)
    ax.bar(x + w/2, en_vals, w, label='EN', color='#DC2626', alpha=0.8)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([PLACE_EN[p][:8] for p in places_a], rotation=45, ha='right', fontsize=7)
    ax.set_title(label, fontsize=11)
    ax.set_ylim(-0.6, 0.4)
    if i == 0:
        ax.legend(fontsize=8)

axes[5].set_visible(False)
fig.suptitle('ABSA: Aspect-Based Sentiment by Place and Language', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. LLM Narrative Summaries

Generated by Qwen2.5-7B-Instruct: one paragraph per place per language group.

In [ ]:
with open(DATA / 'llm_summaries.json', encoding='utf-8') as f:
    summaries = json.load(f)

for place, langs in summaries.items():
    en_name = PLACE_EN.get(place, place)
    print(f'\n{"="*70}')
    print(f'{en_name} ({place})')
    print('='*70)
    print(f'\n  [KO group]: {langs["ko"][:200]}...')
    print(f'\n  [EN group]: {langs["en"][:200]}...')

---
## 10. Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

places_d = [p for p in PLACE_EN if p in df['place'].unique()]
labels = [PLACE_EN[p] for p in places_d]
x = np.arange(len(places_d))
w = 0.35

# 1. Sentiment
ax = axes[0, 0]
ko_s = [float(sent_wide[sent_wide.place==p]['KO_mean']) for p in places_d]
en_s = [float(sent_wide[sent_wide.place==p]['EN_mean']) for p in places_d]
ax.bar(x - w/2, ko_s, w, label='KO', color='#2563EB', alpha=0.8)
ax.bar(x + w/2, en_s, w, label='EN', color='#DC2626', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax.set_title('Sentiment', fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)

# 2. Cross-lingual Similarity
ax = axes[0, 1]
sim_vals = [float(sim[sim.place==p]['ko_en_sim']) for p in places_d]
colors = ['#16a34a' if v >= 0.91 else '#f59e0b' if v >= 0.90 else '#DC2626' for v in sim_vals]
ax.bar(x, sim_vals, 0.5, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax.set_title('Cross-lingual Similarity', fontweight='bold')
ax.set_ylim(0.85, 0.95)

# 3. TTR
ax = axes[1, 0]
ko_t = [float(ttr[(ttr.place==p)&(ttr.language=='ko')]['mean'])
        if not ttr[(ttr.place==p)&(ttr.language=='ko')].empty else 0 for p in places_d]
en_t = [float(ttr[(ttr.place==p)&(ttr.language=='en')]['mean'])
        if not ttr[(ttr.place==p)&(ttr.language=='en')].empty else 0 for p in places_d]
ax.bar(x - w/2, ko_t, w, label='KO', color='#2563EB', alpha=0.8)
ax.bar(x + w/2, en_t, w, label='EN', color='#DC2626', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax.set_title('Lexical Diversity (TTR)', fontweight='bold')
ax.set_ylim(0.75, 1.0)
ax.legend(fontsize=8)

# 4. Motivation comparison
ax = axes[1, 1]
mots = ['Photography & Visual Experience', 'Nature & Scenery',
        'Cultural Heritage & History', 'Food & Gastronomy',
        'Shopping & Entertainment']
mots_short = ['Photo/Visual', 'Nature', 'Cultural', 'Food', 'Shopping']
ko_m = [float(mot_wide.loc[m, 'KO %']) if m in mot_wide.index else 0 for m in mots]
en_m = [float(mot_wide.loc[m, 'EN %']) if m in mot_wide.index else 0 for m in mots]
y = np.arange(len(mots))
h = 0.35
ax.barh(y - h/2, ko_m, h, label='KO', color='#2563EB', alpha=0.8)
ax.barh(y + h/2, en_m, h, label='EN', color='#DC2626', alpha=0.8)
ax.set_yticks(y)
ax.set_yticklabels(mots_short, fontsize=9)
ax.set_title('Travel Motivation (%)', fontweight='bold')
ax.legend(fontsize=8)

fig.suptitle('Cross-Cultural Tourist Review Analysis — Overview',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

**Key Findings:**
1. No overall significant sentiment difference (p=0.656); differences are place-specific
2. Lotte World shows the strongest cultural gap (KO > EN, p=0.036)
3. High cross-lingual similarity (0.89–0.93) across all places
4. Photography/Visual dominates both groups' motivations (~37%)
5. EN reviewers show consistently higher lexical diversity (TTR)
6. ABSA: Crowding is the most negative aspect for both groups
7. KO reviewers show near-zero scores on food and visual aspects (model detection issue)
8. Data collection for 2 additional places (인사동길, N서울타워) is pending